### Tools

#### Models can request to call tools that perform tasks such as fetching data from a database, searching web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.


In [1]:
import os
from langchain_groq import ChatGroq

os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')


model=ChatGroq(model="qwen/qwen3-32b")
response = model.invoke("Hello, how are you?")
response

AIMessage(content='<think>\nOkay, the user asked "Hello, how are you?" which is a greeting. I need to respond in a friendly and welcoming manner. Let me start by acknowledging their greeting and expressing that I\'m doing well. I should also show genuine interest in them by asking about their day or how they\'re feeling.\n\nI want to keep the tone positive and approachable. Maybe add a bit of warmth by using an emoji like a smiley. Let me make sure my response is concise but not too short. Something like, "Hello! I\'m doing well, thank you for asking! 😊 How about you? How\'s your day going so far?" That should work. It\'s friendly, acknowledges their question, and opens the door for further conversation.\n</think>\n\nHello! I\'m doing well, thank you for asking! 😊 How about you? How\'s your day going so far?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 182, 'prompt_tokens': 14, 'total_tokens': 196, 'completion_time': 0.380954545, 'completion_tokens_de

In [2]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather for a location."""
    return f"The weather in {location} is sunny."

model_with_tools = model.bind_tools([get_weather])

In [3]:
response=model_with_tools.invoke("What is the weather like in New York?")
for tool_call in response.tool_calls:
    #View tools calls made by the model
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")
    

Tool:get_weather
Args:{'location': 'New York'}


### Tool Execution Loops

In [6]:
messages = [
    {"role": "user", "content": "What is the weather like in New York?"}
]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response.text)


The weather in New York is currently sunny. A perfect day to enjoy outdoor activities! 🌞


In [7]:
messages

[{'role': 'user', 'content': 'What is the weather like in New York?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in New York. I need to use the get_weather function for that. The function requires a location parameter. New York is the location here, so I should call the function with "New York" as the argument. Let me make sure there\'s no typo. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': '0z8vx31dq', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 155, 'total_tokens': 255, 'completion_time': 0.172693653, 'completion_tokens_details': {'reasoning_tokens': 75}, 'prompt_time': 0.006060953, 'prompt_tokens_details': None, 'queue_time': 0.076691699, 'total_time': 0.178754606}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_efa9879028', 'service